# Recovering an unknown *function*

**Book:** §4.6, Figure 4.3(c) &nbsp;·&nbsp; `ch04/unknown_function_kx.ipynb`

$$\frac{d}{dx}\Big(k(x)\,\frac{du}{dx}\Big) + 1 = 0,\qquad u(0)=u(1)=0,$$

with $k^\star(x)=1+\tfrac12\sin 2\pi x$ **unknown**. Given 30 noisy samples of $u$, recover the
whole conductivity *field* — not a parameter, an infinite-dimensional unknown.

**Formulation.** *Two* networks:

- $u(x) = x(1-x)\,\mathcal N_u(x)$ — hard Dirichlet, both ends;
- $k(x) = \tfrac12 + \mathrm{softplus}\big(\mathcal N_k(x)\big)$ — the softplus guarantees $k>\tfrac12>0$, so the problem stays elliptic. **This positivity constraint is what keeps the inversion well posed.**

$$\mathcal L=\overline{\Big(\tfrac{d}{dx}\big(k\,u_x\big)+1\Big)^2}+50\,\overline{(u(x_i)-u_i^{\rm obs})^2}$$

A classical adjoint method needs a discretised forward solve, an adjoint solve, a regulariser and
a line search. Here it is two networks and one loss.

In [ ]:
import time
import numpy as np, torch, torch.nn as nn
import matplotlib.pyplot as plt
np.random.seed(0); torch.manual_seed(0)
def g1(f, x): return torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]
def mlp(s):
    L = []
    for i in range(len(s)-1):
        L.append(nn.Linear(s[i], s[i+1]))
        if i < len(s)-2: L.append(nn.Tanh())
    return nn.Sequential(*L)
rel = lambda p, e: float(np.sqrt(np.mean((p-e)**2)/np.mean(e**2)))
PI = np.pi

# ---- truth: a fine finite-volume solve, used ONLY to make data and to score ----
kstar = lambda x: 1 + 0.5*np.sin(2*PI*x)
N = 2001; xg = np.linspace(0,1,N); h = xg[1]-xg[0]
kh = kstar(0.5*(xg[:-1]+xg[1:]))
A = np.zeros((N,N)); b = np.ones(N)
for i in range(1, N-1):
    A[i,i-1] = -kh[i-1]/h**2; A[i,i+1] = -kh[i]/h**2; A[i,i] = (kh[i-1]+kh[i])/h**2
A[0,0] = A[-1,-1] = 1.0; b[0] = b[-1] = 0.0
ustar = np.linalg.solve(A, b)

ND = 30
xd = np.sort(np.random.rand(ND)); ud = np.interp(xd, xg, ustar) + 0.002*np.random.randn(ND)
xd_t = torch.tensor(xd, dtype=torch.float32).reshape(-1,1)
ud_t = torch.tensor(ud, dtype=torch.float32).reshape(-1,1)

# ---- two networks: the field, and the unknown coefficient function ----
unet, knet = mlp([1,48,48,1]), mlp([1,32,32,1])
U = lambda x: x*(1-x)*unet(x)                                   # hard Dirichlet
K = lambda x: 0.5 + torch.nn.functional.softplus(knet(x))       # k > 1/2 > 0, always
opt = torch.optim.Adam(list(unet.parameters()) + list(knet.parameters()), 3e-3)

t0 = time.perf_counter()
for e in range(15000):
    if e == 10000:
        for g in opt.param_groups: g['lr'] = 5e-4
    opt.zero_grad()
    x = torch.rand(1000,1).requires_grad_(True)
    res = g1(K(x)*g1(U(x), x), x) + 1.0                         # d/dx(k u_x) + 1 = 0
    ((res**2).mean() + 50*((U(xd_t) - ud_t)**2).mean()).backward()
    opt.step()
print(f'training: {time.perf_counter()-t0:.0f} s')

xt = torch.tensor(xg, dtype=torch.float32).reshape(-1,1)
with torch.no_grad(): kp = K(xt).numpy().ravel()
eK = rel(kp, kstar(xg))
print(f'k(x) rel L2 = {eK:.3f}')

plt.figure(figsize=(8.5,4.5))
plt.plot(xg, kstar(xg), 'g',  lw=2.8, alpha=.6, label=r'true $k^\star(x)$')
plt.plot(xg, kp,        'r--', lw=1.7, label=f'recovered $k(x)$ (rel $L_2$={eK:.3f})')
plt.plot(xg, ustar*8,   'b:',  lw=1.2, alpha=.7, label=r'solution $u$ ($\times 8$)')
plt.scatter(xd, ud*8, c='k', s=16, zorder=5, label=f'{ND} noisy data')
plt.xlabel('x'); plt.legend(fontsize=9); plt.grid(alpha=.3)
plt.title('An unknown coefficient $k(x)$: a whole function recovered from 30 points')
plt.tight_layout(); plt.show()